# max-back-tied-half — ex1: maximum_back with 50/50 tie-splitting

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `max-back-tied-half`. Running the final beacon cell reports progress against the `Backprop: max_back with tied half-mass` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: max_back with tied half-mass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`max-back-tied-half`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "max-back-tied-half"
DD_SUBTOPIC = "Backprop: max_back with tied half-mass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## max_back with tied half-mass — quick refresher

Forward op `out = maximum(x, y)` is elementwise: per position, pick whichever of `x[i]` or `y[i]` is larger. The derivative is **piecewise** and the natural recipe is:

```
dL/dx = grad_out * (x >  y)     # x is the winner
dL/dy = grad_out * (x <  y)     # y is the winner
```

But what about **ties** (`x == y`)? Sending all the mass to one side is asymmetric and discontinuous; sending none is wrong (the sum of partials should equal `grad_out` for the winning value). ARENA's convention is **split the mass 50/50**:

```
bool_sum_x = (x > y) + 0.5 * (x == y)
bool_sum_y = (x < y) + 0.5 * (x == y)
dL/dx = unbroadcast(grad_out * bool_sum_x, x)
dL/dy = unbroadcast(grad_out * bool_sum_y, y)
```

**Invariant.** `bool_sum_x + bool_sum_y == 1` everywhere — gradient mass is conserved across the two inputs, exactly as it should be for an operator whose forward picks ONE of the two values.

### Exercise 1 — maximum_back with 50/50 tie-splitting

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the half-mass tie-splitting convention to derive maximum_back0 and maximum_back1 such that gradient mass is conserved across the two inputs at every position.
> Keywords: maximum-back, ties, half-mass, subgradient, piecewise
> ```

**KCs targeted:** `max-back-tied-half`, `unbroadcast-pattern`

Implement `maximum_back0(grad_out, out, x, y)` and `maximum_back1(grad_out, out, x, y)` for the elementwise op `out = maximum(x, y)`.

Per position, three cases:

1. **`x[i] > y[i]`** — `x` won. `dout/dx = 1`, `dout/dy = 0`.
2. **`x[i] < y[i]`** — `y` won. `dout/dx = 0`, `dout/dy = 1`.
3. **`x[i] == y[i]`** — tie. ARENA convention: **split the mass 50/50** so `dout/dx = 0.5`, `dout/dy = 0.5`.

Recipe:

```python
bool_sum_x = (x > y).float() + 0.5 * (x == y).float()
bool_sum_y = (x < y).float() + 0.5 * (x == y).float()
dL/dx = unbroadcast(grad_out * bool_sum_x, x)
dL/dy = unbroadcast(grad_out * bool_sum_y, y)
```

**Why the tie matters.** A hardcoded `dout/dx = (x > y)` and `dout/dy = (x <= y)` would pass the strict-tie case but is asymmetric (it favours `y`). Half-mass is the only choice that (a) is symmetric in `x` and `y`, and (b) **conserves mass**: `bool_sum_x + bool_sum_y == 1` everywhere, which means `dL/dx + dL/dy == grad_out` at every position (relative to the winning input). This is the right invariant — the forward picks one value, so the backward should distribute the grad onto whoever contributed.

We've also provided `unbroadcast(grad, original)` in the cell — wrap each return so broadcasting between `x` and `y` is handled.

Inputs are plain `torch.Tensor`. No autograd.

In [ ]:
def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad


def maximum_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """dL/dx for out = maximum(x, y), with 50/50 tie-splitting."""
    raise NotImplementedError()


def maximum_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """dL/dy for out = maximum(x, y), with 50/50 tie-splitting."""
    raise NotImplementedError()


def _test_ex1():
    # --- strict case: x[i] > y[i] for all i ---
    x = t.tensor([5.0, 7.0, 9.0])
    y = t.tensor([1.0, 2.0, 3.0])
    out = t.maximum(x, y)
    g0 = maximum_back0(t.ones(3), out, x, y)
    g1 = maximum_back1(t.ones(3), out, x, y)
    assert t.allclose(g0, t.ones(3)), f'x-wins g0: {g0}'
    assert t.allclose(g1, t.zeros(3)), f'x-wins g1: {g1}'

    # --- strict case: y wins ---
    x = t.tensor([1.0, 2.0, 3.0])
    y = t.tensor([5.0, 7.0, 9.0])
    out = t.maximum(x, y)
    g0 = maximum_back0(t.ones(3), out, x, y)
    g1 = maximum_back1(t.ones(3), out, x, y)
    assert t.allclose(g0, t.zeros(3)), f'y-wins g0: {g0}'
    assert t.allclose(g1, t.ones(3)), f'y-wins g1: {g1}'

    # --- pure tie: x == y everywhere, mass splits 50/50 ---
    x = t.tensor([3.0, 3.0, 3.0])
    y = t.tensor([3.0, 3.0, 3.0])
    out = t.maximum(x, y)
    g0 = maximum_back0(t.ones(3), out, x, y)
    g1 = maximum_back1(t.ones(3), out, x, y)
    assert t.allclose(g0, t.full((3,), 0.5)), f'tie g0 (should be 0.5): {g0}'
    assert t.allclose(g1, t.full((3,), 0.5)), f'tie g1 (should be 0.5): {g1}'
    # Conservation: g0 + g1 == grad_out everywhere.
    assert t.allclose(g0 + g1, t.ones(3)), 'mass conservation broke at ties'

    # --- mixed: x wins, y wins, tie all in one tensor ---
    x = t.tensor([5.0, 1.0, 3.0])
    y = t.tensor([1.0, 5.0, 3.0])
    out = t.maximum(x, y)
    grad_out = t.tensor([10.0, 20.0, 40.0])
    g0 = maximum_back0(grad_out, out, x, y)
    g1 = maximum_back1(grad_out, out, x, y)
    assert t.allclose(g0, t.tensor([10.0, 0.0, 20.0])), f'mixed g0: {g0}'
    assert t.allclose(g1, t.tensor([0.0, 20.0, 20.0])), f'mixed g1: {g1}'
    # Conservation at every position.
    assert t.allclose(g0 + g1, grad_out), 'mass conservation across the whole tensor'

    # --- broadcasting: x is (1,4), y is (3,4) ---
    x_b = t.tensor([[1.0, 5.0, 3.0, 8.0]])
    y_b = t.tensor([[3.0, 5.0, 4.0, 2.0],
                    [2.0, 5.0, 1.0, 7.0],
                    [4.0, 5.0, 6.0, 6.0]])
    out_b = t.maximum(x_b, y_b)
    g0_b = maximum_back0(t.ones(3, 4), out_b, x_b, y_b)
    g1_b = maximum_back1(t.ones(3, 4), out_b, x_b, y_b)
    assert g0_b.shape == x_b.shape, f'broadcast g0 shape: {g0_b.shape}'
    assert g1_b.shape == y_b.shape, f'broadcast g1 shape: {g1_b.shape}'
    # Column-1 is all ties (x=5, y=5) → 0.5 contribution from EACH of 3 rows = 1.5 on x's slot.
    assert t.allclose(g0_b[0, 1], t.tensor(1.5)), f'tie column on x: {g0_b[0, 1]}'

    # --- per-position conservation pre-unbroadcast ---
    # At positions in (out_b == x_b) ∩ (out_b == y_b) we should see g0+g1 sum to grad_out,
    # even though after unbroadcast the per-position numbers shift.
    raw_g0 = t.ones(3, 4) * ((x_b > y_b).float() + 0.5 * (x_b == y_b).float())
    raw_g1 = t.ones(3, 4) * ((x_b < y_b).float() + 0.5 * (x_b == y_b).float())
    assert t.allclose(raw_g0 + raw_g1, t.ones(3, 4)), 'half-mass invariant violated'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def maximum_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # x's share of the grad: 1 where x > y, 0 where x < y, 0.5 at ties.
    bool_sum = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return unbroadcast(grad_out * bool_sum, x)


def maximum_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # y's share: 1 where x < y, 0 where x > y, 0.5 at ties.
    bool_sum = (x < y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return unbroadcast(grad_out * bool_sum, y)
```

**Why half-mass is the right convention.** The subgradient of `max(x, y)` at `x == y` is the whole convex hull of the two one-sided derivatives — any `λ * 1_x + (1-λ) * 1_y` for `λ ∈ [0, 1]` is valid. `λ = 0.5` is the only choice that's symmetric in `x` and `y` and treats ties as the limit of the averaging procedure.

**Mass conservation as the diagnostic.** `bool_sum_x + bool_sum_y == 1` everywhere — if your implementation breaks this (e.g. uses `(x >= y)` and `(x <= y)`, which double-count ties to 1+1=2), the reverse pass over-counts. The mass-conservation test in the harness catches this directly.

**`.to(grad_out.dtype)` casts.** `(x > y)` returns a `bool` tensor; multiplying it by a float still works in torch, but the intermediate `bool_sum` would have weird dtypes (mixing bool and float). Explicit `.to(grad_out.dtype)` keeps everything in the same float space and avoids surprise promotions.

**ReLU as a special case.** `relu(x) = maximum(x, 0)` — so `relu_back` is `maximum_back0` with `y = 0`. The tie-splitting convention means `relu_back(0)` returns `0.5 * grad_out` (controversial but consistent with this library).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()